In [ ]:
## FINETUNE, TEST, MERGE
print("Installing required libraries...")
!pip install -q -U "torch==2.3.1" "transformers==4.41.2" "peft==0.11.1" "accelerate==0.30.1" "trl==0.9.4" "datasets==2.19.2" "bitsandbytes==0.43.1"

import json
import os
import torch
from datasets import Dataset
from peft import LoraConfig, PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, BitsAndBytesConfig
from trl import SFTTrainer
from huggingface_hub import notebook_login

INSTRUCTION = (
        "You are a Python code refactoring tool for NumPy. "
        "Your task is to replace only the deprecated functions in the given code snippet with their modern equivalents. "
        "Do not change the code's logic, indentation, or add any new functionality. "
        "Respond ONLY with the new code. "
        "If no functions are deprecated, return the original code."
    )

print("Preparing dataset")
PATH_TO_TRAINING = 'training_data.json'
with open(PATH_TO_TRAINING, 'r', encoding='utf-8') as f:
    training_data = json.load(f)

def create_prompt(sample):
    return (
        f"<start_of_turn>user\n{INSTRUCTION}\n\n"
        f"### INPUT CODE:\n```python\n{sample['input']}\n```<end_of_turn>\n"
        f"<start_of_turn>model\n```python\n{sample['output']}\n```<end_of_turn>"
    )

dataset = Dataset.from_list([{'text': create_prompt(s)} for s in training_data])
print("Dataset prepared.")

# Training
notebook_login()

base_model_id = "google/codegemma-2b"
adapter_save_name = "codegemma-2b-libsmart-finetuned"

# save GPU memory using 4bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(base_model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

#configure model
model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map={"": 0}
)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

training_args = TrainingArguments(
    output_dir="./models",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    logging_steps=10,
    fp16=True, # Must be true for 4-bit training
    optim="paged_adamw_8bit", # Must be paged for 4-bit training
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_args,
    packing=True,
)

print("Starting fine-tuning")
trainer.train()
print("Fine-tuning DONE!")

print("Merging with base model")

adapter_path = f"./{adapter_save_name}"
trainer.model.save_pretrained(adapter_path)
del model
del trainer
torch.cuda.empty_cache()
print(f"LoRA adapter saved to '{adapter_path}'")

#merge adapter with base model
print(f"---Loading base model '{base_model_id}' in FP16 for merging")
base_model_fp16 = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)
print("Base model loaded in FP16.")

print(f"--- Applying LoRA adapter")
model_merged_fp16 = PeftModel.from_pretrained(base_model_fp16, adapter_path)
model_merged_fp16 = model_merged_fp16.merge_and_unload()
print("DONE Adapter merged.")

# Save de-quantized model for gguf conversion
merged_model_dir = f"{adapter_save_name}-merged-fp16"
model_merged_fp16.save_pretrained(merged_model_dir)
tokenizer.save_pretrained(merged_model_dir)
print(f"DONE Clean de-quantized model saved to: {merged_model_dir}")


In [ ]:
#CONVERT TO GGUF AND QUANTIZE


# Build llama.cpp using cmake
print("\nBuilding llama.cpp")
!git clone https://github.com/ggerganov/llama.cpp.git

!cmake -S llama.cpp -B llama.cpp/build -DLLAMA_CUBLAS=OFF -DGGML_CUDA=ON

# build 
!cmake --build llama.cpp/build --target all -j 2

# Convert FP16 model to GGUF:
QUANTIZATION_METHOD = "q4_k_m"
print(f"\nConverting and quantizing to {QUANTIZATION_METHOD}...")
fp16_gguf_path = f"./{merged_model_dir}.fp16.gguf"
!python llama.cpp/convert_hf_to_gguf.py ./{merged_model_dir} \
    --outfile {fp16_gguf_path} \
    --outtype f16

final_gguf_path = f"./{adapter_save_name}.{QUANTIZATION_METHOD}.gguf"

# quantize
!./llama.cpp/build/bin/llama-quantize {fp16_gguf_path} {final_gguf_path} {QUANTIZATION_METHOD}


print("\n" + "="*50)
print(f "DONE: GGUF CONVERSION COMPLETE!")
print(f"Quantized model saved as: {final_gguf_path}")
print("="*50)


In [ ]:
#TEST FUNCTIONALITY OF GGUF FILE

import subprocess
import os
import sys

# --- Final GGUF Model Test ---

print("Final Test of GGUF Model:")

try:
    final_gguf_path = f"./{adapter_save_name}.{QUANTIZATION_METHOD}.gguf"
    main_executable_path = subprocess.check_output(["find", "llama.cpp", "-type", "f", "-name", "llama-cli"]).decode("utf-8").strip()

    if not os.path.exists(final_gguf_path) or not main_executable_path:
        raise FileNotFoundError("GGUF file or 'llama-cli' executable not found.")

    # Run inference
    test_inputs = [
        "       arr = np.array([val], dtype=np.int)",
        "transposed = np.fastCopyAndTranspose(arr)",
        "       unique_elements = np.unique1d([1, 2, 1, 3, 2])",
    ]

    for i, code in enumerate(test_inputs):
        prompt_text = (
            f"<start_of_turn>user\n{INSTRUCTION}\n\n"
            f"### INPUT CODE:\n```python\n{code}\n```<end_of_turn>\n"
            f"<start_of_turn>model\n"
        )
        with open("prompt.txt", "w") as f:
            f.write(prompt_text)

        print(f"\nTEST {i+1}:")
        print(f"INPUT:\n```python\n{code}\n```")
        print("\nMODEL OUTPUT:")

        command = [
            main_executable_path,
            "-m", final_gguf_path,
            "-n", "128",
            "-f", "prompt.txt",
            "--color",
            "-ngl", "26",
            "--ignore-eos"
        ]
        
        subprocess.run(command, check=True)

    print("\nAll tests complete.")

except (subprocess.CalledProcessError, FileNotFoundError, NameError) as e:
    print(f"\nAn error occurred during testing: {e}", file=sys.stderr)